In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))

import inventory as I
import manifest as M
import assemble as A
import concepts as C

smh_dir = Path("/home/jormond/Documents/DS217/users/jormond/SMH")
rbh_dir = Path("/home/jormond/Documents/DS217/users/jormond/RBH")
gosh_dir = Path("/home/jormond/Documents/DS217/users/jormond/GOSH")

### What is in the assembled tables
Prints, for each site, every assembled table with its row count and column names (no values). RBH tables are mostly wide (one column per measurement) and GOSH has one file per variable, so these names are what the concept map will be written from.

In [ ]:
assembled_dir = open_picu_dir / "derived" / "assembled"
for site in ["RBH", "GOSH"]:
    C.list_sources(assembled_dir, site)

### Cross-site concept map
Lists the long GOSH label columns first (so GOSH blood gases, fluids and drugs can be mapped), then measures every source in the map at all three sites: stays with a value, rows, share numeric and the 5th, median and 95th percentile. The percentiles show unit differences between sites. Counts and ranges only. This reads every assembled table, so it takes a few minutes.

In [ ]:
C.list_values(assembled_dir, "GOSH", "merged_vent_data__vent_Ventilation_mode_flowsheet_rows",
              "res_value", where=("var_type", "Ventilation"))

In [ ]:
G = "GOSH"
C.list_values(assembled_dir, G, "merged_gas_data__merged_blood_gasses", "standard_variable")
C.list_values(assembled_dir, G, "merged_balance__Net_Bal", "shortLabel")
C.list_values(assembled_dir, G, "merged_balance__Total_fluids", "shortLabel")
C.list_values(assembled_dir, G, "merged_vent_data__vent_Ventilation_mode_flowsheet_rows", "DisplayName")

In [ ]:
pooled_static = pd.read_parquet(open_picu_dir / "derived" / "static" / "pooled_static.parquet")
detail, summary = C.build(assembled_dir, static=pooled_static,
                          save_to=open_picu_dir / "derived" / "concepts" / "concept_map.csv")
pd.set_option("display.max_rows", 100)
print(summary.drop(columns="note").to_string(index=False))

### Coverage by admission year
For a few concepts, the percentage of each site's stays with the concept in each admission year. A source that only exists in some years (for example a system change) shows as a step.

In [ ]:
chk = ["bg_ph", "vent_mode", "tidal_volume", "ie_ratio", "net_balance", "etco2", "abp_sys", "fio2"]
yr, piv = C.by_year(assembled_dir, pooled_static, chk)
print(piv.to_string())

In [ ]:
A.list_tables("GOSH", gosh_dir, include=r"raw_lab_values|lab_components|Archived", exclude="")

In [ ]:
import pyarrow.parquet as pq
pq.ParquetFile(gosh_dir / "gosh_mech_vent_31March2026" / "final_gosh_dataset.parquet").schema.names

In [ ]:
yr, piv = C.by_year(assembled_dir, pooled_static, list(C.CONCEPTS), sites=("GOSH",))

def share(a, b):
    d = yr[yr.year.between(a, b)].groupby("concept")[["stays", "n"]].sum()
    return (100 * d.stays / d.n).round(1)

era = pd.DataFrame({"pre_epic_pct": share(2013, 2018), "post_epic_pct": share(2020, 2022)})
print(era[(era.pre_epic_pct >= 50) & (era.post_epic_pct < 0.5 * era.pre_epic_pct)].to_string())

In [ ]:
cols = ["hour_timestamp", "pH", "ventMode", "TV_measured", "IERatio"]
old = pd.read_parquet(gosh_dir / "gosh_mech_vent_31March2026" / "final_gosh_dataset.parquet", columns=cols)
old["year"] = pd.to_datetime(old["hour_timestamp"], errors="coerce").dt.year
print(old.groupby("year")[["pH", "ventMode", "TV_measured", "IERatio"]].agg(lambda s: round(100 * s.notna().mean(), 1)).to_string())

### Does the old GOSH parquet hold post-2019 data we are missing?
`final_gosh_dataset.parquet` is an older hourly dataset built from the GOSH sources. For every column, this shows how many hourly rows have a value before and after the CareVue to Epic switch (about 20 April 2019). If blood gas, vent mode, tidal volume or I:E ratio have values after 2019 here, they exist in a source we have not assembled. Counts only.

In [ ]:
old_path = gosh_dir / "gosh_mech_vent_31March2026" / "final_gosh_dataset.parquet"
era = C.era_columns(old_path)
print(era.sort_values("post_pct").to_string())

In [ ]:
g = pooled_static[pooled_static.site == "GOSH"]
print((pd.to_datetime(g["t0"]) >= "2019-04-20").sum(), len(g))

### Which features does each stay have?
Builds a stay-by-feature table (has a value, or not) so users can choose a feature set and the stays that have it. The pattern of available features could itself identify a stay, so this also counts how many stays share their pattern with fewer than 10 others, per feature and per group. Row-level: saved in derived/ only.

In [ ]:
avail = C.availability(assembled_dir, pooled_static,
                       save_to=open_picu_dir / "derived" / "concepts" / "stay_availability.parquet")
print("per feature:", C.pattern_uniqueness(avail))
print("per group:  ", C.pattern_uniqueness(avail, by_group=True))

In [ ]:
summary = pd.read_csv(open_picu_dir / "derived" / "concepts" / "concept_summary.csv")
avail = pd.read_parquet(open_picu_dir / "derived" / "concepts" / "stay_availability.parquet")

In [ ]:
for pct in (90, 70, 50):
    cols = C.survivors(summary, pct)["concept"].tolist()
    print(pct, len(cols), C.pattern_uniqueness(avail[cols]))
cols = C.survivors(summary, 50, exclude_era=True)["concept"].tolist()
print("50, era-safe", len(cols), C.pattern_uniqueness(avail[cols]))

In [ ]:
cols = C.survivors(summary, 50)["concept"].tolist()
print(C.rare_share(avail[cols], pooled_static, by=["site"]))
print(C.rare_share(avail[cols], pooled_static, by=["site", "year"]).unstack("site"))

## Drugs: names in the data

The concept map has no drugs yet. The SMH drug IDs in `smh_mappings.py` were taken from the extraction SQL, and four infusion IDs were added by reading `Drug_terse`. Before mapping drugs across sites, list the drug names in the SMH medication, infusion and antibiotic tables, with the number of stays each appears in, and check them against the ID maps. Counts of stays only; no rows leave.

In [ ]:
assembled_dir = open_picu_dir / "derived" / "assembled"
C.list_sources(assembled_dir, "SMH", show_columns=True)

Number of stays per drug name, for each SMH drug table. Names with fewer than 10 stays are not listed.

In [ ]:
for t in ["medications", "infusions"]:
    print(t)
    print(C.list_values(assembled_dir, "SMH", t, "Drug_terse", min_stays=10, top=80))

In [ ]:
C.list_values(assembled_dir, "SMH", "infusions", "Drug_terse", min_stays=1, top=80)

In [ ]:
import pandas as pd
m = pd.read_parquet(assembled_dir / "SMH" / "ai_picu_medications_v2.parquet", columns=["Drug_terse", "utcChartTime"])
f = m[m["Drug_terse"] == "Furosemide"]["utcChartTime"].astype(str)
print(f.str.len().value_counts().head())
t = pd.to_datetime(f, errors="coerce", format="mixed")
print(t.isna().sum(), t.dt.month.max(), t.dt.day.max())

## Drugs: RBH and GOSH

Same check as for SMH. First find the drug tables and the column that holds the drug name at RBH and GOSH, then list the names with the number of stays each appears in. The aim is a cross-site drug table with merged spellings (for example the two morphine spellings at SMH). RBH has `infusion`, `meds`, `sedation` and `sedation_old` (the last two overlap in time); GOSH has `filtered_ALL_drugs` and `intrv_details` (pre-Epic) and `medsG_medication_admins` (Epic only), so the drug lists are expected to differ by era.

In [ ]:
for S in ["RBH", "GOSH"]:
    C.list_sources(assembled_dir, S, show_columns=True)

Number of stays per drug name for each RBH drug table. Names seen in fewer than 10 stays are not listed.

In [ ]:
for t in ["ai_picu_infusion", "ai_picu_meds", "ai_picu_sedation", "ai_picu_sedation_old"]:
    print(t)
    C.list_values(assembled_dir, "RBH", t, "Drug", min_stays=10, top=80)

Same for GOSH. The drug files are `filtered_ALL_drugs_merged`, `filtered_CI_drugs_merged` (8.1M rows, probably continuous infusions) and `intrv_details`, which name the drug in `shortLabel`; `medsG_medication_admins` (Epic only) and the antibiotics table use `drug_name`. The pre-Epic and Epic tables are listed separately, because names and coverage differ by era.

In [ ]:
_D = "merged_drugs__"
for t, col in [(_D + "filtered_ALL_drugs_merged", "shortLabel"),
               (_D + "filtered_CI_drugs_merged", "shortLabel"),
               (_D + "intrv_details", "shortLabel"),
               (_D + "medsG_medication_admins", "drug_name"),
               ("July2026extraction_by_alex__pAIC__merged_antibiotics__antibiotics", "drug_name")]:
    print(t)
    C.list_values(assembled_dir, "GOSH", t, col, min_stays=10, top=80)

Where does each GOSH drug source start and stop? Number of stays per year for the main drugs in the pre-Epic infusion table (`filtered_CI_drugs_merged`) and the Epic-only administration table (`medsG_medication_admins`). Years with fewer than 10 stays are blanked. If the infusion table stops at the Epic switch (April 2019), GOSH has no infusion rates after it.

In [ ]:
_D = "merged_drugs__"
C.names_by_year(assembled_dir, "GOSH", _D + "filtered_CI_drugs_merged", "shortLabel", "observation_datetime", top=8)

In [ ]:
C.names_by_year(assembled_dir, "GOSH", _D + "medsG_medication_admins", "drug_name", "observation_datetime", top=8)

In [ ]:
C.names_by_year(assembled_dir, "GOSH", _D + "filtered_ALL_drugs_merged", "shortLabel", "observation_datetime", top=8)

## Drugs: coverage and the GOSH Epic era

Build the drug concepts (any record of the drug in the stay) and measure them like the other features. Then check by admission year whether GOSH coverage drops at the Epic switch (April 2019). Before it the infusion tables hold rates; after it only `medsG` administration records exist, with no rates. A fall in the sedatives and analgesics would mean the GOSH Epic-era infusion data is missing, as for blood gas.

In [ ]:
drugs = [n for n in C.CONCEPTS if n.startswith("drug_")]
detail_d, summary_d = C.build(assembled_dir, static=pooled_static, concepts=drugs)
print(summary_d[["concept", "RBH_pct", "SMH_pct", "GOSH_pct"]].to_string())

Percent of GOSH stays with each drug, by admission year. A step down from 2019 means the Epic-era records do not capture the drug.

In [ ]:
out, piv = C.by_year(assembled_dir, pooled_static, drugs, sites=["GOSH"])
print(piv.round(0).to_string())

Check whether the RBH `IE_Ratio` and `HFOV_Frequency` columns hold real values or a default fill. Both show 100% of stays. List the most common values with the number of stays each appears in, and compare the ratio columns (`Insp_Ratio`, `Exp_Ratio`).

In [ ]:
for c in ["IE_Ratio", "HFOV_Frequency", "Insp_Ratio", "Exp_Ratio"]:
    C.list_values(assembled_dir, "RBH", "ai_picu_vent", c, min_stays=10, top=8)

In [ ]:
print(sorted(p.name for p in (open_picu_dir / "notebooks").glob("*.ipynb")))

# common feature sets

In [ ]:
for m in [90, 70, 50]:
    print(m, len(C.survivors(summary, m)), len(C.survivors(summary, m, exclude_era=True)))

## availability and patterns

In [ ]:
derived_dir = open_picu_dir / "derived"
avail = C.availability(assembled_dir, pooled_static, save_to=derived_dir / "concepts" / "stay_availability.parquet")
print(avail.shape)
for m in [90, 70, 50]:
    cols = C.survivors(summary, m)["concept"].tolist()
    print(m, len(cols), C.pattern_uniqueness(avail[cols]))
cols = C.survivors(summary, 50, exclude_era=True)["concept"].tolist()
print("50 era-safe", len(cols), C.pattern_uniqueness(avail[cols]))
print("by group", C.pattern_uniqueness(avail, by_group=True))